# Baseline Evaluation: kg-axel (Single-Shot Generation)

**Purpose**: Run baseline kg-axel system and compute comprehensive metrics

**Output**: `outputMetrics/baseline_kg_axel.csv`

## Baseline Configuration:
- System: kg-axel
- Prompt: Chain-of-Thought (CoT)
- Schema: only_paths
- Generation: Single-shot (no refinement)

## Metrics Computed:
- **Cypher Similarity**: BLEU, Rouge-L, Jaro, Jaccard
- **Output Similarity**: Pass@1 Output, Jaccard Output
- **Validators**: Syntax, Schema, Properties
- **Derived Scores**: Pass@1 Score, KG Validity Score, Jaccard Output Score, JaRou Score
- **Composite**: LLMetric-Q

## 1. Setup

In [ ]:
import sys
import os
import time
import csv
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print(f"Notebook started: {datetime.now()}")
print(f"Working directory: {Path.cwd()}")

## 2. Import Modules

In [ ]:
# Baseline system (kg-axel)
from baseline.kg_axel_generator import KGAxelGenerator

# Database executor
from system.graph_executor import GraphExecutor

# Comprehensive metrics calculator
from evaluation.comprehensive_metrics import ComprehensiveMetricsCalculator, create_metrics_dataframe

# Utilities
from utils.schema_loader import load_schema

print("Modules imported successfully")

## 3. Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    "name": "baseline_kg_axel",
    "model": os.getenv("DEFAULT_MODEL", "qwen/qwen-2.5-coder-32b-instruct"),
    "prompt_type": "cot",  # Chain-of-Thought
    "schema_type": "only_paths",  # Best config from kg-axel
    "temperature": float(os.getenv("TEMPERATURE", 0.0)),
    "max_tokens": int(os.getenv("MAX_TOKENS", 512)),
    "rate_limit_delay": float(os.getenv("RATE_LIMIT_DELAY", 2.0)),
    "batch_size": int(os.getenv("BATCH_SIZE", 10)),
    "batch_pause": float(os.getenv("BATCH_PAUSE", 15.0)),
}

# Paths
OUTPUT_DIR = Path.cwd().parent / "outputMetrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = f"{CONFIG['name']}.csv"
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILENAME

print("Baseline Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nOutput: {OUTPUT_PATH}")

## 4. Load Data

In [ ]:
# Load ground truth questions
gt_file = Path.cwd().parent / "data" / "ground_truth" / "ground_truth_52.csv"

questions = []
with open(gt_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        questions.append({
            "id": i + 1,
            "question": row["Pertanyaan"],
            "ground_truth": row["Cypher Query"],
            "complexity": row["Tingkat Kompleksitas"],
            "reasoning_level": row["Tingkat Penalaran"],
            "sublevel": row["Sublevel"]
        })

print(f"Loaded {len(questions)} questions")

# Show distribution
from collections import Counter
print(f"\nBy Complexity: {dict(Counter(q['complexity'] for q in questions))}")
print(f"By Reasoning: {dict(Counter(q['reasoning_level'] for q in questions))}")
print(f"By Sublevel: {dict(Counter(q['sublevel'] for q in questions))}")

## 5. Initialize Systems

In [ ]:
# Load schema
schema = load_schema(CONFIG["schema_type"])
print(f"Schema loaded: {CONFIG['schema_type']} ({len(schema)} chars)")

# Initialize kg-axel generator (CoT + only_paths)
baseline = KGAxelGenerator(
    model=CONFIG["model"],
    temperature=CONFIG["temperature"],
    max_tokens=CONFIG["max_tokens"],
    prompt_type=CONFIG["prompt_type"]
)

# Initialize graph executor
try:
    executor = GraphExecutor()
    print("Graph executor initialized successfully")
except Exception as e:
    print(f"Warning: Could not initialize graph executor - {e}")
    executor = None

# Initialize comprehensive metrics calculator
metrics_calculator = ComprehensiveMetricsCalculator(executor=executor)
print("Metrics calculator initialized")

print(f"\nBaseline system ready:")
print(f"  Model: {CONFIG['model']}")
print(f"  Prompt: {CONFIG['prompt_type']}")
print(f"  Schema: {CONFIG['schema_type']}")

## 6. Run Baseline Inference & Evaluation

In [ ]:
from IPython.display import clear_output

# Storage for results
results = []

def update_display(current, total, q_id, tokens, llmetric_q):
    """Update progress display."""
    clear_output(wait=True)
    pct = current / total * 100
    
    # Calculate running statistics
    total_tokens_so_far = sum(r.get("total_tokens", 0) for r in results)
    avg_llmetric = sum(r.get("LLMetric-Q", 0) for r in results) / current if current > 0 else 0
    pass_at_1_count = sum(1 for r in results if r.get("Pass@1 Output"))
    
    print(f"Progress: {current}/{total} ({pct:.1f}%)")
    print(f"Last: Q{q_id} - tokens={tokens}, LLMetric-Q={llmetric_q:.1f}")
    print(f"")
    print(f"Running Statistics:")
    print(f"  Pass@1 Output: {pass_at_1_count}/{current} ({100*pass_at_1_count/current:.1f}%)")
    print(f"  Avg LLMetric-Q: {avg_llmetric:.2f}")
    print(f"  Total Tokens: {total_tokens_so_far:,}")

print("Starting Baseline Inference & Evaluation...")
print("=" * 60)
start_time = datetime.now()

for i, q in enumerate(questions):
    # Batch pause
    if i > 0 and i % CONFIG["batch_size"] == 0:
        print(f"\n[Batch pause: {CONFIG['batch_pause']}s]")
        time.sleep(CONFIG["batch_pause"])
    
    try:
        # Generate query using kg-axel
        generated_query, response = baseline.generate(
            question=q["question"],
            schema=schema
        )
        
        # Compute all metrics
        metrics = metrics_calculator.compute_all_metrics(
            question_id=q["id"],
            question=q["question"],
            ground_truth_query=q["ground_truth"],
            generated_query=generated_query,
            prompt_technique="CoT",  # Chain-of-Thought
            schema_format=CONFIG["schema_type"],
            reasoning_level=q["reasoning_level"],
            sublevel=q["sublevel"],
            complexity=q["complexity"]
        )
        
        # Add token usage
        metrics["total_tokens"] = response.usage["total_tokens"]
        metrics["input_tokens"] = response.usage["prompt_tokens"]
        metrics["output_tokens"] = response.usage["completion_tokens"]
        
        results.append(metrics)
        
        update_display(
            i + 1, len(questions), q["id"],
            metrics["total_tokens"], metrics["LLMetric-Q"]
        )
        
    except Exception as e:
        print(f"Error on Q{q['id']}: {e}")
        # Create minimal error entry
        error_metrics = {
            "ID Pertanyaan": q["id"],
            "Teknik Prompt Engineering": "CoT",
            "Format Representasi Skema KG": CONFIG["schema_type"],
            "Tingkat Penalaran": q["reasoning_level"],
            "Sublevel": q["sublevel"],
            "Tingkat Kompleksitas": q["complexity"],
            "Cypher LLM": "",
            "LLMetric-Q": 0.0,
            "total_tokens": 0,
            "input_tokens": 0,
            "output_tokens": 0
        }
        results.append(error_metrics)
    
    # Rate limiting
    if i < len(questions) - 1:
        time.sleep(CONFIG["rate_limit_delay"])

end_time = datetime.now()
duration = str(end_time - start_time)

print(f"\n\nBaseline evaluation completed!")
print(f"Duration: {duration}")

## 7. Save Results

In [ ]:
import pandas as pd

# Create DataFrame with proper column ordering
df = create_metrics_dataframe(results)

# Add token columns at the end if not already there
if "total_tokens" not in df.columns:
    df["total_tokens"] = [r.get("total_tokens", 0) for r in results]
    df["input_tokens"] = [r.get("input_tokens", 0) for r in results]
    df["output_tokens"] = [r.get("output_tokens", 0) for r in results]

# Save to CSV
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

## 8. Summary Statistics

In [ ]:
print("=" * 60)
print("BASELINE EVALUATION SUMMARY (kg-axel + CoT)")
print("=" * 60)

total = len(df)

print(f"\nConfiguration:")
print(f"  Model: {CONFIG['model']}")
print(f"  Prompt: {CONFIG['prompt_type']}")
print(f"  Schema: {CONFIG['schema_type']}")

print(f"\nPerformance Metrics:")
print(f"  Pass@1 Output: {df['Pass@1 Output'].sum()}/{total} ({100*df['Pass@1 Output'].mean():.1f}%)")
print(f"  Pass@1 Score (avg): {df['Pass@1 Score'].mean():.2f}")
print(f"  KG Validity Score (avg): {df['KG Validity Score'].mean():.2f}")
print(f"  Jaccard Output Score (avg): {df['Jaccard Output Score'].mean():.2f}")
print(f"  JaRou Score (avg): {df['JaRou Score'].mean():.2f}")
print(f"  LLMetric-Q (avg): {df['LLMetric-Q'].mean():.2f}")

print(f"\nSimilarity Metrics (avg):")
print(f"  BLEU: {df['BLEU'].mean():.2f}")
print(f"  Rouge-L F1: {df['Rouge-L F1-score'].mean():.2f}")
print(f"  Jaro Similarity: {df['Jaro Similarity'].mean():.2f}")
print(f"  Jaccard Similarity: {df['Jaccard Similarity'].mean():.2f}")

print(f"\nValidator Results:")
syntax_valid = df['Syntax Validator'].sum()
schema_valid = (df['Schema Validator'] == 1.0).sum()
props_valid = ((df['Properties Validator'] == 1.0) | df['Properties Validator'].isna()).sum()
print(f"  Syntax Valid: {syntax_valid}/{total} ({100*syntax_valid/total:.1f}%)")
print(f"  Schema Valid: {schema_valid}/{total} ({100*schema_valid/total:.1f}%)")
print(f"  Properties Valid: {props_valid}/{total} ({100*props_valid/total:.1f}%)")

print(f"\nCost & Performance:")
total_tokens = df['total_tokens'].sum()
print(f"  Total tokens: {total_tokens:,}")
print(f"  Avg tokens/question: {total_tokens/total:,.0f}")
print(f"  Duration: {duration}")

print(f"\nOutput: {OUTPUT_PATH}")
print("=" * 60)

## 9. Preview Results

In [ ]:
# Preview key columns
preview_cols = [
    "ID Pertanyaan", "Tingkat Kompleksitas", "Sublevel",
    "Pass@1 Output", "LLMetric-Q", "Pass@1 Score", "KG Validity Score"
]
display(df[preview_cols].head(10))